# Modelos de ML (IV-J)
Clasificación de bandas salariales (Random Forest, XGBoost opcional) con baseline léxico estático (Ampolu) vs. + métricas de red; y K-Means ponderado por I(h).

In [5]:
#%pip install -q scikit-learn xgboost openpyxl

In [6]:
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

of = pd.read_excel("ofertas_limpias_skills.xlsx")
of["lista"] = of["skills"].fillna("").map(lambda s: [x.strip() for x in s.split(";") if x.strip()])
inf = pd.read_csv("skill_influencia.csv").set_index("habilidad")
skills = sorted(inf.index)

# Variable objetivo: bandas por cuantiles (3) sobre ofertas con salario declarado
con = of[of.sal_mid.notna()].reset_index(drop=True)
y = pd.qcut(con.sal_mid, 3, labels=False)

# Baseline léxico estático (presencia de skill + seniority) vs. + métricas de red
X_base = pd.concat([pd.DataFrame([[int(s in l) for s in skills] for l in con.lista], columns=skills),
                    pd.get_dummies(con.seniority, prefix="sen").astype(int)], axis=1)
agg = lambda col, f: con.lista.map(lambda l: f([inf.loc[s, col] for s in l if s in inf.index] or [0]))
red = pd.DataFrame({"n_skills": con.lista.map(len),
                    "I_sum": agg("I", sum), "I_mean": agg("I", np.mean), "I_max": agg("I", max),
                    "PR_mean": agg("pagerank", np.mean), "Auth_mean": agg("authority", np.mean)})
X_red = pd.concat([X_base, red], axis=1)
print("Ofertas con salario:", len(con), "| clases:", y.value_counts().sort_index().tolist())

Ofertas con salario: 191 | clases: [71, 68, 52]


In [7]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)   # validación cruzada estratificada

def evaluar(modelo, X):
    r = []
    for tr, te in cv.split(X, y):
        p = clone(modelo).fit(X.iloc[tr], y.iloc[tr]).predict(X.iloc[te])
        r.append((accuracy_score(y.iloc[te], p), f1_score(y.iloc[te], p, average="macro"),
                  np.mean(np.abs(y.iloc[te].values - p) <= 1)))
    return np.mean(r, axis=0)

modelos = {"Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)}
try:
    from xgboost import XGBClassifier
    modelos["XGBoost"] = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, eval_metric="mlogloss", random_state=42)
except ImportError:
    print("xgboost no instalado: se omite")

filas = []
for nm, m in modelos.items():
    for fe, X in [("Baseline léxico (Ampolu)", X_base), ("Léxico + métricas de red", X_red)]:
        a, f, pm = evaluar(m, X)
        filas.append((nm, fe, a, f, pm))
met = pd.DataFrame(filas, columns=["modelo", "features", "exactitud", "f1_macro", "error_pm1_banda"])
met.to_csv("modelos_metricas.csv", index=False); print(met.round(3))

rf = clone(modelos["Random Forest"]).fit(X_red, y)
pd.Series(rf.feature_importances_, X_red.columns).sort_values(ascending=False).head(10)

          modelo                  features  exactitud  f1_macro  \
0  Random Forest  Baseline léxico (Ampolu)      0.548     0.547   
1  Random Forest  Léxico + métricas de red      0.531     0.530   
2        XGBoost  Baseline léxico (Ampolu)      0.525     0.521   
3        XGBoost  Léxico + métricas de red      0.512     0.506   

   error_pm1_banda  
0            0.926  
1            0.916  
2            0.919  
3            0.905  


Auth_mean                0.109176
I_sum                    0.093060
PR_mean                  0.090189
I_mean                   0.083238
sen_practicante          0.062465
n_skills                 0.057523
I_max                    0.055153
costos y presupuestos    0.030814
sen_no especificado      0.027924
mantenimiento            0.022489
dtype: float64

In [8]:
# --- Clustering: K-Means sobre el vector de habilidades ponderado por I(h) ---
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

al = of[of.lista.map(len) > 0].reset_index(drop=True)
Xw = pd.DataFrame([[inf.I[s] * (s in l) for s in skills] for l in al.lista], columns=skills)

sil = {k: silhouette_score(Xw, KMeans(k, n_init=10, random_state=42).fit_predict(Xw)) for k in range(2, 9)}
K = max(sil, key=sil.get); print("Silhouette:", {k: round(v, 3) for k, v in sil.items()}, "-> K =", K)
al["cluster"] = KMeans(K, n_init=10, random_state=42).fit_predict(Xw)
al[["pc1", "pc2"]] = PCA(2, random_state=42).fit_transform(Xw)

res = []
for c, g in al.groupby("cluster"):
    top = Xw.loc[g.index].mean().sort_values(ascending=False).head(6).index.tolist()
    res.append((c, len(g), ", ".join(top), g.seniority.mode()[0], g.sal_mid.median()))
cl = pd.DataFrame(res, columns=["cluster", "n_ofertas", "skills_top", "seniority_modal", "salario_mediano"])
al[["url", "titulo", "cluster", "pc1", "pc2"]].to_csv("clusters.csv", index=False); cl.to_csv("cluster_resumen.csv", index=False)
cl

Silhouette: {2: 0.159, 3: 0.147, 4: 0.087, 5: 0.133, 6: 0.098, 7: 0.092, 8: 0.104} -> K = 2


,cluster,n_ofertas,skills_top,seniority_modal,salario_mediano
0,0,410,"mantenimiento, comunicación, autocad, ingles, ...",no especificado,2400.0
1,1,223,"excel, kpi, costos y presupuestos, mantenimien...",no especificado,2000.0
